In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# 1. Dataset (XOR)
# -------------------------------------------------
X = np.array([
    [1, 1],
    [-1, -1],
    [1, -1],
    [-1, 1]
], dtype=float)

y = np.array([1, 1, -1, -1], dtype=float)

n = len(y)

# -------------------------------------------------
# 2. Kernel function
# K(x,z) = (1 + x1*z1)(1 + x2*z2)
# -------------------------------------------------
def kernel(X, Z):
    return (1 + X[:, 0:1] * Z[:, 0]) * (1 + X[:, 1:2] * Z[:, 1])

# Compute Gram matrix
K = kernel(X, X)

# -------------------------------------------------
# 3. Solve dual with simple projected gradient ascent
# -------------------------------------------------
alpha = np.zeros(n)
lr = 0.01

for _ in range(5000):
    gradient = 1 - (y * (K @ (alpha * y)))
    alpha += lr * gradient
    
    # Projection: alpha >= 0
    alpha = np.maximum(alpha, 0)
    
    # Enforce constraint sum(alpha_i y_i) = 0
    alpha -= y * (np.sum(alpha * y) / np.sum(y**2))

# -------------------------------------------------
# 4. Compute bias term b
# -------------------------------------------------
support = alpha > 1e-4
b = np.mean(
    y[support] - (K[support] @ (alpha * y))
)

# -------------------------------------------------
# 5. Prediction function
# -------------------------------------------------
def predict(X_new):
    K_new = kernel(X, X_new)
    return np.sign((alpha * y) @ K_new + b)

# -------------------------------------------------
# 6. Plot decision boundary
# -------------------------------------------------
xx, yy = np.meshgrid(
    np.linspace(-2, 2, 200),
    np.linspace(-2, 2, 200)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = predict(grid)
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X[:, 0], X[:, 1], c=y, s=100)
plt.title("Kernel SVM (Pure NumPy)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()